# Water Potability — Exploration rapide

Ce notebook conserve uniquement les étapes d'EDA essentielles à la compréhension du problème avant l'entraînement (voir `src/train.py` pour le pipeline complet). L'EDA exhaustive du script R d'origine (histogrammes, corrélations par classe, PCA, tests de Wilcoxon...) n'est pas reproduite ici : ce projet est centré sur le ML et l'application, pas sur l'exploration statistique.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_raw_data, get_missing_values_report, class_balance

df = load_raw_data()
df.head()

## 1. Valeurs manquantes

Quatre variables contiennent des NA : `ph`, `tds`, `sulfate`, `conductivity`. Elles seront imputées par la médiane au sein du pipeline (voir `src/train.py`), jamais avant le split train/test.

In [ ]:
get_missing_values_report(df)

## 2. Déséquilibre des classes

Le dataset est déséquilibré (~61% NonPotable / ~39% Potable selon la source des données). Cela justifie l'usage de `class_weight="balanced"` pour les modèles linéaires et le SVM lors de l'entraînement.

In [ ]:
class_balance(df["potability"])

## 3. Distribution des variables par classe

Aperçu rapide pour repérer les variables qui séparent le mieux les deux classes.

In [ ]:
features = [c for c in df.columns if c != "potability"]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, feature in zip(axes.flat, features):
    sns.boxplot(data=df, x="potability", y=feature, ax=ax)
    ax.set_title(feature)
plt.tight_layout()
plt.show()

## 4. Corrélations entre variables

Vérifie l'absence de colinéarité forte entre les variables explicatives.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.show()

## Conclusion

Aucune variable ne sépare parfaitement les classes prise isolément, et les corrélations entre variables restent faibles : cela justifie l'usage d'un modèle non linéaire (SVM RBF) capable de capturer des frontières de décision complexes, conformément à la conclusion du benchmark réalisé dans `src/train.py`.